# Building Reliable Data Lakes with Apache Spark

# Building Lakehouses with Apache Spark and Delta Lake

In [1]:
# https://docs.delta.io/quick-start/#python

import pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from delta import *

spark_conf = SparkConf()
spark_conf = spark_conf.setAppName('spark-delta-lake')
# spark_conf = spark_conf.set("spark.jars", "../jars/delta-spark_2.12-3.3.2.jar")
# spark_conf = spark_conf.set("spark.driver.extraClassPath", "../jars/delta-spark_2.12-3.3.2.jar")
# spark_conf = spark_conf.set("spark.jars.packages", "io.delta:delta-spark_2.12:3.3.2")
spark_conf = spark_conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
spark_conf = spark_conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
spark = configure_spark_with_delta_pip(SparkSession.builder.config(conf=spark_conf)).getOrCreate()
print(spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-48ddf575-d8be-40aa-914b-1ec635a89e51;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 116ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

3.5.8


## Loading Data into a Delta Lake Table

In [2]:
sourcePath = "../data/learning-spark-v2/loans/loan-risks.snappy.parquet"
# Configure Delta Lake path
deltaPath = "/tmp/loans_delta"
# Create the Delta Lake table with the same loans data
(spark.read.format("parquet").load(sourcePath)
 .write.format("delta").save(deltaPath))
# Create a view on the data called loans_delta
spark.read.format("delta").load(deltaPath).createOrReplaceTempView("loans_delta")

In [3]:
spark.sql("SELECT count(*) FROM loans_delta").show()

26/04/21 09:10:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+
|count(1)|
+--------+
|   14705|
+--------+



In [4]:
spark.sql("SELECT * FROM loans_delta LIMIT 5").show()

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
|      0|       1000|   182.22|        CA|
|      1|       1000|   361.19|        WA|
|      2|       1000|   176.26|        TX|
|      3|       1000|   1000.0|        OK|
|      4|       1000|   249.98|        PA|
+-------+-----------+---------+----------+



## Loading Data Streams into a Delta Lake Table

SKIP

## Enforcing Schema on Write to Prevent Data Corruption

In [7]:
# from pyspark.sql.functions import *
# cols = ['loan_id', 'funded_amnt', 'paid_amnt', 'addr_state', 'closed']
# items = [
#   (1111111, 1000, 1000.0, 'TX', True),
#   (2222222, 2000, 0.0, 'CA', False)
# ]
# loanUpdates = (spark.createDataFrame(items, cols)
#   .withColumn("funded_amnt", col("funded_amnt").cast("int")))
# loanUpdates.write.format("delta").mode("append").save(deltaPath)



# AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: af3922b9-5e45-47e3-a489-752ae79d1f95).
# To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
# '.option("mergeSchema", "true")'.
# For other operations, set the session configuration
# spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
# specific to the operation for details.

# Table schema:
# root
# -- loan_id: long (nullable = true)
# -- funded_amnt: integer (nullable = true)
# -- paid_amnt: double (nullable = true)
# -- addr_state: string (nullable = true)


# Data schema:
# root
# -- loan_id: long (nullable = true)
# -- funded_amnt: integer (nullable = true)
# -- paid_amnt: double (nullable = true)
# -- addr_state: string (nullable = true)
# -- closed: boolean (nullable = true)

## Evolving Schemas to Accommodate Changing Data

In [6]:
from pyspark.sql.functions import *

cols = ['loan_id', 'funded_amnt', 'paid_amnt', 'addr_state', 'closed']
items = [
  (1111111, 1000, 1000.0, 'TX', True),
  (2222222, 2000, 0.0, 'CA', False)
]
loanUpdates = (spark.createDataFrame(items, cols)
  .withColumn("funded_amnt", col("funded_amnt").cast("int")))

(loanUpdates.write.format("delta").mode("append")
.option("mergeSchema", "true") # merge schema
.save(deltaPath))

In [11]:
spark.sql("SELECT * FROM loans_delta WHERE loan_id IN ('1111111','2222222')").show()

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
|1111111|       1000|   1000.0|        TX|
|2222222|       2000|      0.0|        CA|
+-------+-----------+---------+----------+



## Transforming Existing Data

In [ ]:
# Updating data to fix errors
spark.sql("SELECT * FROM loans_delta WHERE addr_state = 'OR' ORDER BY loan_id LIMIT 1").show()

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
|     39|       1400|   1400.0|        OR|
+-------+-----------+---------+----------+



In [ ]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, deltaPath)
deltaTable.update("addr_state = 'OR'", {"addr_state": "'WA'"})

In [17]:
spark.sql("SELECT * FROM loans_delta WHERE loan_id = '39'").show()

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
|     39|       1400|   1400.0|        WA|
+-------+-----------+---------+----------+



In [18]:
# Deleting user-related data
spark.sql("SELECT * FROM loans_delta WHERE funded_amnt >= paid_amnt").show(1)

deltaTable = DeltaTable.forPath(spark, deltaPath)
deltaTable.delete("funded_amnt >= paid_amnt")

spark.sql("SELECT * FROM loans_delta WHERE funded_amnt >= paid_amnt").show(1)

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
|      0|       1000|   182.22|        CA|
+-------+-----------+---------+----------+
only showing top 1 row

+-------+-----------+---------+----------+
|loan_id|funded_amnt|paid_amnt|addr_state|
+-------+-----------+---------+----------+
+-------+-----------+---------+----------+



In [19]:
# Upserting change data to a table using merge()
(deltaTable
  .alias("t")
  .merge(loanUpdates.alias("s"), "t.loan_id = s.loan_id")
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute())

In [ ]:
# Deduplicating data while inserting using insert-only merge

# (deltaTable
#   .alias("t")
#   .merge(historicalUpdates.alias("s"), "t.loan_id = s.loan_id")
#   .whenNotMatchedInsertAll()
#   .execute())

## Auditing Data Changes with Operation History

In [20]:
deltaTable.history().show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2026-04-21 09:22:...|  NULL|    NULL|    MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/3.5....|
|      3|2026-04-21 09:22:...|  NULL|    NULL|   DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/3.5....|
|      2|2

In [21]:
(deltaTable
  .history(3)
  .select("version", "timestamp", "operation", "operationParameters")
  .show(truncate=False))

+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                                      |
+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-04-21 09:22:52.82 |MERGE    |{predicate -> ["(loan_id#3991L = loan_id#1225L)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|3      |2026-04-21 09:22:16.497|DELETE   |{predicate -> ["(cast(funded_amnt#399

## Querying Previous Snapshots of a Table with Time Travel

In [28]:
(spark.read
  .format("delta")
  .option("timestampAsOf", "2026-04-21 09:22:52.82")
  .load(deltaPath)).show()

+-------+-----------+---------+----------+------+
|loan_id|funded_amnt|paid_amnt|addr_state|closed|
+-------+-----------+---------+----------+------+
|1111111|       1000|   1000.0|        TX|  true|
|2222222|       2000|      0.0|        CA| false|
+-------+-----------+---------+----------+------+



In [27]:
(spark.read.format("delta")
  .option("versionAsOf", "4")
  .load(deltaPath)).show()

+-------+-----------+---------+----------+------+
|loan_id|funded_amnt|paid_amnt|addr_state|closed|
+-------+-----------+---------+----------+------+
|1111111|       1000|   1000.0|        TX|  true|
|2222222|       2000|      0.0|        CA| false|
+-------+-----------+---------+----------+------+



# Cleanup

In [30]:
spark.stop()